# Week 3 — Data Preprocessing
**California Property Close Price Prediction**

> Deliverable: `02_preprocessing.ipynb` + cleaned CSV
>
> Handbook Week 3: missing-value handling / categorical encoding / numeric scaling / chronological train-test split (test = most recent month, train = X preceding months, X tunable)
>

> Also enforces **AVM Best Practices**: chronological split (no random shuffling), remove impossible records with counts,
> outlier thresholds learned on train only, leakage features excluded (ListPrice etc.), all preprocessing fit on train only.

In [1]:
# 0. Setup
from pathlib import Path
import glob, warnings
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ----  local data ----
# Example for 28 files: /Users/wy/Desktop/idx excahnge/data science/dataset/csv/CRMLSSold*.csv
DATA_GLOB = '/Users/wy/Desktop/idx excahnge/data science/dataset/csv/CRMLSSold*.csv'
OUTPUT_DIR = Path('output_week3')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Candidate training windows X (months). We have 28 months (202401-202604), test=202604; try several and pick the best by Week-4 baseline R2
TRAIN_WINDOW_CHOICES = [5, 11, 17, 26]
DEFAULT_X = 11  # default: one-year window; shorter windows may win when market regimes shift -- decide by experiment

print('Config OK ->', OUTPUT_DIR.resolve())

Config OK -> /Users/wy/Desktop/idx excahnge/data science/pythonProject/output_week3


## 1. Load data
- Handbook requires at least 6 months; we have 28 (202401–202604), so load everything.
- Filename note: the `_filled` suffix has no special meaning -- just inconsistent naming (confirmed); it does not affect processing.
- Work on local copies only; never modify the raw FTP folder.

In [2]:
files = sorted(glob.glob(DATA_GLOB))
assert len(files) >= 6, 'Handbook requires at least 6 months of data'

In [3]:
# Load everything (MLS exports mix dtypes, so low_memory=False)
dfs = []
for f in files:
    d = pd.read_csv(f, low_memory=False)
    d['__source_file'] = Path(f).name
    dfs.append(d)
df = pd.concat(dfs, ignore_index=True)
print('Raw shape:', df.shape)
df.head(2).T

Raw shape: (615707, 85)


,0,1
Flooring,"Carpet,Tile,Wood",NaN
ViewYN,True,NaN
WaterfrontYN,NaN,NaN
BasementYN,NaN,NaN
PoolPrivateYN,False,NaN
...,...,...
__source_file,CRMLSSold202401_filled.csv,CRMLSSold202401_filled.csv
BuyerAgentAOR,NaN,NaN
ListAgentAOR,NaN,NaN
OriginatingSystemName,NaN,NaN


## 2. Core filters (required by Task Prompt)
Keep only `PropertyType == 'Residential'` and `PropertySubType == 'SingleFamilyResidence'`.

In [4]:
initial_n = len(df)
for c in ['PropertyType', 'PropertySubType']:
    df[c] = df[c].astype(str).str.strip()
mask = (df['PropertyType'] == 'Residential') & (df['PropertySubType'] == 'SingleFamilyResidence')
df = df[mask].copy()
print(f'{initial_n} -> {len(df)} rows ({len(df)/initial_n:.1%} retained)')

615707 -> 309735 rows (50.3% retained)


## 3. Data quality: remove records that cannot be true (Best Practices §02)
Log the count at every step. 

In [5]:
clean_log = []
def log_step(name, before):
    after = len(df)
    r = before - after
    clean_log.append((name, before, after, r))
    print(f'{name}: {before} -> {after}, removed {r}')

In [6]:
# 3a. Target must exist and be > 0 (never impute the target)
before = len(df)
df['ClosePrice'] = pd.to_numeric(df['ClosePrice'], errors='coerce')
df = df[df['ClosePrice'].notna() & (df['ClosePrice'] > 0)].copy()
log_step('Valid ClosePrice', before)

Valid ClosePrice: 309735 -> 309732, removed 3


In [7]:
# 3b. CloseDate must parse (the chronological split depends on it)
before = len(df)
df['CloseDate'] = pd.to_datetime(df['CloseDate'], errors='coerce')
df = df[df['CloseDate'].notna()].copy()
df['CloseMonth'] = df['CloseDate'].dt.to_period('M')
log_step('Valid CloseDate', before)

Valid CloseDate: 309732 -> 309732, removed 0


In [8]:
# 3c. Logical impossibilities (tailored to the 2026-04 sample)
before = len(df)
for c in ['LivingArea', 'LotSizeArea']:  # area <= 0 is impossible
    s = pd.to_numeric(df[c], errors='coerce')
    bad = (s <= 0) & s.notna()
    print(f'  {c} <= 0: {bad.sum()} rows')
    df = df[~bad].copy()

  LivingArea <= 0: 126 rows
  LotSizeArea <= 0: 163 rows


In [9]:
# Invalid lat/long: Latitude==0 (missing-value placeholder), Longitude>0 (CA longitudes are negative) -> set to NaN, impute later
lat = pd.to_numeric(df['Latitude'], errors='coerce')
lon = pd.to_numeric(df['Longitude'], errors='coerce')
print(f'  Latitude==0: {(lat == 0).sum()} rows, Longitude>0: {(lon > 0).sum()} rows -> set to NaN')
df['Latitude'] = lat.mask(lat == 0)
df['Longitude'] = lon.mask(lon > 0)

  Latitude==0: 22 rows, Longitude>0: 25 rows -> set to NaN


In [10]:
# close before listing is impossible
ld = pd.to_datetime(df['ListingContractDate'], errors='coerce')  
bad = ld.notna() & (df['CloseDate'] < ld)
print(f'  CloseDate < ListingContractDate: {bad.sum()} rows')
df = df[~bad].copy()

  CloseDate < ListingContractDate: 44 rows


In [11]:
log_step('Logical-impossibility filters', before)

Logical-impossibility filters: 309732 -> 309399, removed 333


In [12]:
# 3d. Deduplicate: same ListingKey + CloseDate = same transaction
before = len(df)
n_dup = df.duplicated(subset=['ListingKey', 'CloseDate']).sum()
print(f'  Duplicates: {n_dup} rows')
df = df.drop_duplicates(subset=['ListingKey', 'CloseDate']).copy()

  Duplicates: 29 rows


In [13]:
log_step('Deduplication', before)

Deduplication: 309399 -> 309370, removed 29


In [14]:
log_df = pd.DataFrame(clean_log, columns=['step', 'before', 'after', 'removed'])
log_df.to_csv(OUTPUT_DIR / 'cleaning_log.csv', index=False)
print(log_df.to_string(index=False))

                         step  before  after  removed
             Valid ClosePrice  309735 309732        3
              Valid CloseDate  309732 309732        0
Logical-impossibility filters  309732 309399      333
                Deduplication  309399 309370       29


## 4. Column policy: leakage-only drop (Best Practices §04)

This implements the instructor's rule: use only information available for any single-family address at prediction time, on- or off-market. **Timing self-check**: on valuation day, before any sale outcome is known, could you have this value?


| Group | Columns | Action | Reason |
|---|---|---|---|
| Leaking: price fields | `ListPrice`, `OriginalListPrice`, `DaysOnMarket` | Drop | Near-perfect correlation with ClosePrice; unavailable for off-market homes |
| Leaking: process dates | `ListingContractDate`, `PurchaseContractDate`, `ContractStatusChangeDate` | Drop | Exist only because the home was listed/sold  |
| Leaking: tax-assessed | `TaxAnnualAmount`, `TaxYear` | Drop | Assessed values  |
| Leaking: agent/brokerage | 16 agent/office columns (`ListAgent*`, `BuyerAgent*`, `CoListAgent*`, `*OfficeName`, `*AOR`, `ListAgentEmail`, `BuyerAgentMlsId`) | Drop | Can proxy pricing strategy rather than intrinsic value  |
| Not in this data | price-reduction count/flag, cumulative DOM, settlement/commission fields | N/A | No such columns in the CRMLSSold files — nothing to drop |


In [15]:
DROP_LEAKAGE = [
    # Near-perfect correlation with ClosePrice; unavailable for off-market homes
    'ListPrice', 'OriginalListPrice', 'DaysOnMarket',
    # Sale-process artifacts: only exist because the home was listed/sold.
    # ListingContractDate was already used for the cleaning check in Section 3.
    'ListingContractDate', 'PurchaseContractDate', 'ContractStatusChangeDate',
    # Tax-assessed value (also 100% missing in this data)
    'TaxAnnualAmount', 'TaxYear',
    # Listing agent / brokerage: can proxy pricing strategy, not intrinsic value
    'ListAgentAOR', 'ListAgentEmail', 'ListAgentFirstName', 'ListAgentLastName',
    'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName',
    'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
    'CoBuyerAgentFirstName', 'ListOfficeName', 'BuyerOfficeName',
    'CoListOfficeName', 'BuyerAgentAOR', 'BuyerOfficeAOR',
]

drop_cols = [c for c in DROP_LEAKAGE if c in df.columns]
print(f'Dropped {len(drop_cols)} leakage columns')

y = df['ClosePrice'].copy()
X = df.drop(columns=['ClosePrice'] + drop_cols, errors='ignore')
print('X shape:', X.shape, '| y shape:', y.shape)
print('Remaining columns:', list(X.columns))

Dropped 24 leakage columns
X shape: (309370, 61) | y shape: (309370,)
Remaining columns: ['Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'ListingKey', 'CloseDate', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'YearBuilt', 'BuyerAgencyCompensationType', 'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City', 'BuyerAgencyCompensation', 'BuildingAreaTotal', 'BedroomsTotal', 'ElementarySchoolDistrict', 'BelowGradeFinishedArea', 'BusinessType', 'StateOrProvince', 'CoveredSpaces', 'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories', 'HighSchool', 'Levels', 'LotSizeDimensions', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', '

## 5. Chronological split (Best Practices §01)
- **test** = most recent complete month (2026-04); touch it only once, at the end
- **validation** = second-most-recent month (2026-03), for tuning in Week 5+; never tune on test
- **train** = X months before test, X tunable (`TRAIN_WINDOW_CHOICES`); compare Week-4 baseline R2 on the **same test** month to pick the best X
- No random splits: shuffling leaks future market conditions into training and overstates performance

In [16]:
months = sorted(X['CloseMonth'].dropna().unique())
print(f'{len(months)} months: {months[0]} ~ {months[-1]}')
test_month, val_month = months[-1], months[-2]
print('test:', test_month, '| validation:', val_month)

28 months: 2024-01 ~ 2026-04
test: 2026-04 | validation: 2026-03


In [17]:
X_test, y_test = X[X['CloseMonth'] == test_month].copy(), y[X['CloseMonth'] == test_month].copy()
X_rest, y_rest = X[X['CloseMonth'] != test_month].copy(), y[X['CloseMonth'] != test_month].copy()

In [18]:
print('\nCandidate X windows (test fixed at', test_month, '):')
for win in TRAIN_WINDOW_CHOICES:
    ms = months[max(0, len(months) - 1 - win):-1]
    n = (X_rest['CloseMonth'].isin(ms)).sum()
    print(f'  X={win}: {ms[0]}..{ms[-1]} ({len(ms)} months) -> {n} rows')


Candidate X windows (test fixed at 2026-04 ):
  X=5: 2025-11..2026-03 (5 months) -> 48154 rows
  X=11: 2025-05..2026-03 (11 months) -> 118589 rows
  X=17: 2024-11..2026-03 (17 months) -> 179330 rows
  X=26: 2024-02..2026-03 (26 months) -> 288585 rows


In [19]:
train_months = months[max(0, len(months) - 1 - DEFAULT_X):-1]
tr_mask = X_rest['CloseMonth'].isin(train_months)
X_train, y_train = X_rest[tr_mask].copy(), y_rest[tr_mask].copy()
X_val = X_train[X_train['CloseMonth'] == val_month].copy(); y_val = y_train[X_train['CloseMonth'] == val_month].copy()
X_tr = X_train[X_train['CloseMonth'] != val_month].copy(); y_tr = y_train[X_train['CloseMonth'] != val_month].copy()
print(f'\nX={DEFAULT_X}: model-train {len(X_tr)} | val({val_month}) {len(X_val)} | test({test_month}) {len(X_test)}')


X=11: model-train 107037 | val(2026-03) 11552 | test(2026-04) 12471


## 6. Outliers: learn thresholds on train only (Best Practices §03)
Take the 0.5th / 99.5th percentiles of train ClosePrice, freeze them, and apply unchanged to val/test. **Never recompute on test**.

In [20]:
lo, hi = y_tr.quantile(0.005), y_tr.quantile(0.995)
print(f'Train-only cutoffs: 0.5%=${lo:,.0f}, 99.5%=${hi:,.0f} (frozen for val/test)')

Train-only cutoffs: 0.5%=$187,000, 99.5%=$8,750,000 (frozen for val/test)


In [21]:
def apply_cut(X_, y_, name):
    m = (y_ >= lo) & (y_ <= hi)
    print(f'{name}: {len(y_)} -> {m.sum()} (removed {len(y_) - m.sum()})')
    return X_[m].copy(), y_[m].copy()

In [22]:
X_tr, y_tr = apply_cut(X_tr, y_tr, 'model-train')

model-train: 107037 -> 105974 (removed 1063)


In [23]:
X_val, y_val = apply_cut(X_val, y_val, 'validation ')

validation : 11552 -> 11416 (removed 136)


In [24]:
X_test, y_test = apply_cut(X_test, y_test, 'test')

test: 12471 -> 12339 (removed 132)


## 7. Missing values: handle deliberately (Best Practices §06)
- Audit missingness on **train** only before deciding
- Drop threshold: >60% 
- Important features: median/mode imputation; add a `_was_missing` indicator where missingness itself is informative (e.g. missing lot size ~= condo signal)
- Never impute the target

In [25]:
miss = X_tr.isna().mean().sort_values(ascending=False)
print('Train missing-rate Top15:'); print((miss.head(15) * 100).round(1).to_string())

Train missing-rate Top15:
FireplacesTotal                 100.0
lonfilled                       100.0
BuyerAgencyCompensation         100.0
ElementarySchoolDistrict        100.0
MiddleOrJuniorSchoolDistrict    100.0
latfilled                       100.0
AboveGradeFinishedArea          100.0
BusinessType                    100.0
BuyerAgencyCompensationType     100.0
CoveredSpaces                   100.0
WaterfrontYN                     99.9
BelowGradeFinishedArea           99.3
BasementYN                       97.6
BuilderName                      95.3
LotSizeDimensions                93.7


In [26]:
miss

FireplacesTotal                 1.0
lonfilled                       1.0
BuyerAgencyCompensation         1.0
ElementarySchoolDistrict        1.0
MiddleOrJuniorSchoolDistrict    1.0
                               ... 
PropertyType                    0.0
__source_file                   0.0
CloseDate                       0.0
ListingKey                      0.0
CloseMonth                      0.0
Length: 61, dtype: float64

In [27]:
MISS_TH = 0.60
drop_m = miss[miss > MISS_TH].index.tolist()
# LotSizeArea (mixed units per metadata) and LotSizeAcres (exact duplicate of sqft) dropped; keep LotSizeSquareFeet
# Row identifiers excluded: unique (or near-unique) per listing, so they carry no generalizable signal for unseen addresses.
# UnparsedAddress is near-unique free text; its useful location signal is already captured by structured geo features
# (City, CountyOrParish, PostalCode, Latitude/Longitude), and raw addresses are PII.
EXTRA_DROP = ['ListingKey', 'ListingKeyNumeric', 'ListingId', 'UnparsedAddress','LotSizeArea', 'LotSizeAcres']
drop_m += [c for c in EXTRA_DROP if c in X_tr.columns and c not in drop_m]

In [28]:
# Core features are kept even above threshold -> impute + indicator, with a warning
for c in ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet','Latitude', 'Longitude']:
    if c in drop_m: drop_m.remove(c); print(f'WARNING: core feature {c} above threshold but kept; using imputation + indicator')
print(f'Dropping {len(drop_m)} high-missingness/unuseful columns:', drop_m)
X_tr = X_tr.drop(columns=drop_m); X_val = X_val.drop(columns=drop_m); X_test = X_test.drop(columns=drop_m)
# Drop zero-variance columns (single-valued columns carry no information)
zero_var = [c for c in X_tr.columns if X_tr[c].nunique(dropna=False) <= 1]
print('Zero-variance columns:', zero_var)
X_tr = X_tr.drop(columns=zero_var); X_val = X_val.drop(columns=zero_var); X_test = X_test.drop(columns=zero_var)

Dropping 29 high-missingness/unuseful columns: ['FireplacesTotal', 'lonfilled', 'BuyerAgencyCompensation', 'ElementarySchoolDistrict', 'MiddleOrJuniorSchoolDistrict', 'latfilled', 'AboveGradeFinishedArea', 'BusinessType', 'BuyerAgencyCompensationType', 'CoveredSpaces', 'WaterfrontYN', 'BelowGradeFinishedArea', 'BasementYN', 'BuilderName', 'LotSizeDimensions', 'BuildingAreaTotal', 'OriginatingSystemName', 'OriginatingSystemSubName', 'ElementarySchool', 'MiddleOrJuniorSchool', 'HighSchool', 'AssociationFeeFrequency', 'SubdivisionName', 'ListingKey', 'ListingKeyNumeric', 'ListingId', 'UnparsedAddress', 'LotSizeArea', 'LotSizeAcres']
Zero-variance columns: ['PropertyType', 'MlsStatus', 'PropertySubType']


## 8. Encoding & scaling (Best Practices §05 / §07)
- Numeric: median imputation + StandardScaler (needed for Week-4 linear regression; harmless for trees)
- Low-cardinality categoricals (<50): mode imputation + one-hot
- High-cardinality categoricals (City 658, PostalCode 1121, etc.): mode imputation + frequency encoding; upgrade to CV target encoding in Week 6
- **Structural leakage guard**: the whole ColumnTransformer is `fit` on train only; `transform` on val/test (section 07)

In [29]:
X_tr.columns

Index(['Flooring', 'ViewYN', 'PoolPrivateYN', 'CloseDate', 'Latitude',
       'Longitude', 'LivingArea', 'MLSAreaMajor', 'CountyOrParish',
       'AttachedGarageYN', 'ParkingTotal', 'YearBuilt', 'StreetNumberNumeric',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'StateOrProvince',
       'FireplaceYN', 'Stories', 'Levels', 'MainLevelBedrooms',
       'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 'PostalCode',
       'AssociationFee', 'LotSizeSquareFeet', '__source_file', 'CloseMonth'],
      dtype='str')

In [30]:
HELPER = ['CloseMonth', '__source_file', 'CloseDate']
feat_cols = [c for c in X_tr.columns if c not in HELPER]
for c in ['LivingArea','BedroomsTotal','BathroomsTotalInteger','LotSizeSquareFeet','YearBuilt',
          'Stories','ParkingTotal','GarageSpaces','MainLevelBedrooms','AssociationFee','Latitude','Longitude']:
    if c in X_tr.columns:
        for X_ in (X_tr, X_val, X_test): X_[c] = pd.to_numeric(X_[c], errors='coerce')
numeric_cols = X_tr[feat_cols].select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in feat_cols if c not in numeric_cols]
print(f'Numeric ({len(numeric_cols)}):', numeric_cols)
print(f'Categorical ({len(cat_cols)}):', [(c, int(X_tr[c].nunique())) for c in cat_cols])
LOW = [c for c in cat_cols if X_tr[c].nunique() < 50]
HIGH = [c for c in cat_cols if c not in LOW]
print('Low-cardinality (one-hot):', LOW); print('High-cardinality (frequency):', HIGH)

Numeric (13): ['Latitude', 'Longitude', 'LivingArea', 'ParkingTotal', 'YearBuilt', 'StreetNumberNumeric', 'BathroomsTotalInteger', 'BedroomsTotal', 'Stories', 'MainLevelBedrooms', 'GarageSpaces', 'AssociationFee', 'LotSizeSquareFeet']
Categorical (13): [('Flooring', 253), ('ViewYN', 2), ('PoolPrivateYN', 2), ('MLSAreaMajor', 1003), ('CountyOrParish', 59), ('AttachedGarageYN', 2), ('City', 943), ('StateOrProvince', 4), ('FireplaceYN', 2), ('Levels', 17), ('NewConstructionYN', 2), ('HighSchoolDistrict', 414), ('PostalCode', 1909)]
Low-cardinality (one-hot): ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'StateOrProvince', 'FireplaceYN', 'Levels', 'NewConstructionYN']
High-cardinality (frequency): ['Flooring', 'MLSAreaMajor', 'CountyOrParish', 'City', 'HighSchoolDistrict', 'PostalCode']


In [31]:
FLAG_MISS_TH = 0.01  # build missing indicators only for numeric cols with >1% missing in train
FLAG_COLS = [c for c in numeric_cols if X_tr[c].isna().mean() > FLAG_MISS_TH]
FLAG_COLS

['Stories',
 'MainLevelBedrooms',
 'GarageSpaces',
 'AssociationFee',
 'LotSizeSquareFeet']

In [32]:
for c in FLAG_COLS:
    for X_ in (X_tr, X_val, X_test): X_[f'{c}_was_missing'] = X_[c].isna().astype(int)
num_final = numeric_cols + [f'{c}_was_missing' for c in FLAG_COLS]
print('Missing-indicator columns:', [f'{c}_was_missing' for c in FLAG_COLS])

Missing-indicator columns: ['Stories_was_missing', 'MainLevelBedrooms_was_missing', 'GarageSpaces_was_missing', 'AssociationFee_was_missing', 'LotSizeSquareFeet_was_missing']


In [33]:
from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Simple leakage-free encoding for high-cardinality cats (fit sees train only)."""
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.cols_ = list(X.columns)
        self.maps_ = {c: X[c].value_counts(normalize=True) for c in self.cols_}
        return self
    def transform(self, X):
        X = pd.DataFrame(X, columns=self.cols_)
        out = pd.DataFrame(index=X.index)
        for c in self.cols_:
            out[c] = X[c].map(self.maps_[c]).fillna(0.0)
        return out.values
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = getattr(self, 'cols_', [])
        return np.asarray(list(input_features))


In [34]:
preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_final),
    ('lowcard', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                         ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01, sparse_output=False))]), LOW),
    ('highcard', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('freq', FrequencyEncoder())]), HIGH),
], remainder='drop', verbose_feature_names_out=False)

X_tr_enc = preprocess.fit_transform(X_tr)   # fit on train ONLY
X_val_enc = preprocess.transform(X_val)
X_test_enc = preprocess.transform(X_test)
print('Encoded shapes:', X_tr_enc.shape, X_val_enc.shape, X_test_enc.shape)
print('Residual NaNs:', int(np.isnan(X_tr_enc).sum() + np.isnan(X_val_enc).sum() + np.isnan(X_test_enc).sum()))

Encoded shapes: (105974, 41) (11416, 41) (12339, 41)
Residual NaNs: 0


## 9. Save artifacts
- `cleaned_*.csv`: human-readable pre-encoding version (handbook deliverable)
- `*.npy` + `preprocessing_pipeline.joblib`: load directly for Week-4 modeling
- `reproducibility_notes.txt`: data snapshot, splits, thresholds, random seed (section 11)

In [35]:
for X_, y_, name in [(X_tr, y_tr, 'train'), (X_val, y_val, 'val'), (X_test, y_test, 'test')]:
    out = X_.copy(); out['ClosePrice'] = y_.values; out['__split'] = name
    out.to_csv(OUTPUT_DIR / f'cleaned_{name}.csv', index=False)
    print(f'cleaned_{name}.csv:', out.shape)
pd.concat([X_tr.assign(ClosePrice=y_tr.values, __split='train'),
           X_val.assign(ClosePrice=y_val.values, __split='val'),
           X_test.assign(ClosePrice=y_test.values, __split='test')],
          ignore_index=True).to_csv(OUTPUT_DIR / 'cleaned_all.csv', index=False)
np.save(OUTPUT_DIR / 'X_train_enc.npy', X_tr_enc); np.save(OUTPUT_DIR / 'X_val_enc.npy', X_val_enc)
np.save(OUTPUT_DIR / 'X_test_enc.npy', X_test_enc)
np.save(OUTPUT_DIR / 'y_train.npy', y_tr.values); np.save(OUTPUT_DIR / 'y_val.npy', y_val.values)
np.save(OUTPUT_DIR / 'y_test.npy', y_test.values)
joblib.dump(preprocess, OUTPUT_DIR / 'preprocessing_pipeline.joblib')
pd.Series(list(preprocess.get_feature_names_out())).to_csv(OUTPUT_DIR / 'feature_list.csv', index=False, header=['feature'])
notes_lines = [
    f'Data: {DATA_GLOB} ({len(files)} files)',
    f'Rows: raw {initial_n} -> train {len(X_tr)} / val {len(X_val)} / test {len(X_test)}',
    f'test={test_month}, val={val_month}, train window X={DEFAULT_X} (candidates {TRAIN_WINDOW_CHOICES})',
    f'Outlier cutoffs (train-only, frozen): ${lo:,.0f} ~ ${hi:,.0f}',
    'Missingness drop threshold: >{:.0%}'.format(MISS_TH),
    f'Random seed: {RANDOM_STATE}',
]
(OUTPUT_DIR / 'reproducibility_notes.txt').write_text(chr(10).join(notes_lines))
print(chr(10).join(notes_lines)); print('Artifacts saved ->', OUTPUT_DIR.resolve())

cleaned_train.csv: (105974, 36)
cleaned_val.csv: (11416, 36)
cleaned_test.csv: (12339, 36)
Data: /Users/wy/Desktop/idx excahnge/data science/dataset/csv/CRMLSSold*.csv (28 files)
Rows: raw 615707 -> train 105974 / val 11416 / test 12339
test=2026-04, val=2026-03, train window X=11 (candidates [5, 11, 17, 26])
Outlier cutoffs (train-only, frozen): $187,000 ~ $8,750,000
Missingness drop threshold: >60%
Random seed: 42
Artifacts saved -> /Users/wy/Desktop/idx excahnge/data science/pythonProject/output_week3


## 10. Final checks

In [36]:
assert X_tr_enc.shape[1] == X_test_enc.shape[1] == X_val_enc.shape[1], 'Feature count mismatch across train/val/test!'
for y_, n in [(y_tr, 'train'), (y_val, 'val'), (y_test, 'test')]:
    print(f'{n}: median ${y_.median():,.0f}  mean ${y_.mean():,.0f}  range ${y_.min():,.0f}~${y_.max():,.0f}')
print('\nWeek 3 deliverables ready: this notebook + output_week3/cleaned_*.csv')
print('Week 4: load X_train_enc.npy / y_train.npy + pipeline, train a Linear Regression baseline, report R2 on test')

train: median $884,000  mean $1,184,566  range $187,000~$8,750,000
val: median $890,000  mean $1,197,444  range $187,000~$8,300,000
test: median $915,000  mean $1,234,695  range $189,000~$8,700,000

Week 3 deliverables ready: this notebook + output_week3/cleaned_*.csv
Week 4: load X_train_enc.npy / y_train.npy + pipeline, train a Linear Regression baseline, report R2 on test
